# 01_extract_data.ipynb - Ekstraksi Data dari Multi-Port Database

Notebook ini untuk:
- Extract data dari 3 port database (3306, 3307, 3308)
- Apply namespace P1/P2/P3 untuk menghindari collision ID
- Normalize datetime SFA (UTC → WIB)
- Simpan ke Parquet untuk caching

**ATURAN:** Notebook ini idempotent - bisa dijalankan ulang dari awal.

In [ ]:
# ── SETUP ────────────────────────────────────────────────────────────────
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "src"))

import pandas as pd
from config import DB_CONFIGS, DATA_DIR, REF_DIR
from db import query_to_df, union_ports, add_port_namespace, normalize_all_datetimes

print("✅ Setup complete")

In [ ]:
# ── EXTRACT: sfa_doccallitem (Kunjungan Sales) ──────────────────────────
# Rule #3: Kolom koordinat bernama szLangitude (TYPO!)
# Rule #5: decDuration dalam DETIK
# Rule #6: dtDocCall harus dinormalisasi

query_doccall = """
SELECT 
    szId,
    szCustomerId,
    szEmployeeId,
    szRouteId,
    dtDocCall,
    decDuration,
    szLangitude,
    szLongitude,
    szBranchId
FROM sfa_doccallitem
WHERE dtDocCall >= DATE_SUB(CURDATE(), INTERVAL 12 MONTH)
"""

print("📋 Extracting sfa_doccallitem from 3 ports...")
df_doccall = union_ports(query_doccall, "port_3306", "port_3307", "port_3308")

# Normalisasi datetime (Rule #6)
df_doccall = normalize_all_datetimes(df_doccall, ['dtDocCall'])

# Tambah namespace (sudah otomatis dari union_ports, tapi kita verifikasi)
print(f"✅ Total records: {len(df_doccall):,}")
print(f"   Sample IDs: {df_doccall['szId'].head(3).tolist()}")

In [ ]:
# ── EXTRACT: sfa_gpstracking (GPS Tracking Sales) ───────────────────────
# Rule #3: Kolom koordinat bernama szLangitude (TYPO!)

query_gps = """
SELECT 
    szId,
    szEmployeeId,
    dtTimestamp,
    szLangitude,
    szLongitude,
    decSpeed
FROM sfa_gpstracking
WHERE dtTimestamp >= DATE_SUB(CURDATE(), INTERVAL 12 MONTH)
"""

print("📋 Extracting sfa_gpstracking from 3 ports...")
df_gps = union_ports(query_gps, "port_3306", "port_3307", "port_3308")

# Normalisasi datetime
df_gps = normalize_all_datetimes(df_gps, ['dtTimestamp'])

print(f"✅ Total records: {len(df_gps):,}")

In [ ]:
# ── EXTRACT: dms_sm_saledtl (Transaksi Penjualan) ───────────────────────
# Sumber: SCHEMA_PORT_3306.csv

query_sales = """
SELECT 
    szId,
    szCustomerId,
    szDocId,
    dtDoc,
    szItemId,
    decQty,
    decPrice,
    decAmount,
    szBranchId
FROM dms_sm_saledtl
WHERE dtDoc >= DATE_SUB(CURDATE(), INTERVAL 12 MONTH)
"""

print("📋 Extracting dms_sm_saledtl from 3 ports...")
df_sales = union_ports(query_sales, "port_3306", "port_3307", "port_3308")

# Normalisasi datetime
df_sales = normalize_all_datetimes(df_sales, ['dtDoc'])

print(f"✅ Total records: {len(df_sales):,}")

In [ ]:
# ── EXTRACT: dms_sm_addressinfo (Alamat & Koordinat Outlet) ─────────────
# Rule #4: Kolom koordinat bernama szLatitude (tanpa typo)

query_address = """
SELECT 
    szCustomerId,
    szCustomerName,
    szAddress,
    szLatitude,
    szLongitude,
    szCity,
    szProvince,
    szBranchId
FROM dms_sm_addressinfo
"""

print("📋 Extracting dms_sm_addressinfo from 3 ports...")
df_address = union_ports(query_address, "port_3306", "port_3307", "port_3308")

print(f"✅ Total records: {len(df_address):,}")

In [ ]:
# ── EXTRACT: mst_customer (Master Data Outlet) ──────────────────────────

query_customer = """
SELECT 
    szCustomerId,
    szCustomerName,
    szClass,
    szStatus,
    szSalesmanId,
    szRouteId,
    szBranchId
FROM mst_customer
"""

print("📋 Extracting mst_customer from 3 ports...")
df_customer = union_ports(query_customer, "port_3306", "port_3307", "port_3308")

print(f"✅ Total records: {len(df_customer):,}")

In [ ]:
# ── EXTRACT: mst_employee (Master Data Sales) ───────────────────────────

query_employee = """
SELECT 
    szEmployeeId,
    szEmployeeName,
    szPosition,
    szBranchId
FROM mst_employee
WHERE szPosition LIKE '%sales%' OR szPosition LIKE '%merchandiser%'
"""

print("📋 Extracting mst_employee from 3 ports...")
df_employee = union_ports(query_employee, "port_3306", "port_3307", "port_3308")

print(f"✅ Total records: {len(df_employee):,}")

In [ ]:
# ── SIMPAN KE PARQUET (Caching) ─────────────────────────────────────────
# Rule #7: Semua DataFrame sudah punya namespace dari union_ports()

print("💾 Saving to Parquet...")

DATA_DIR.mkdir(parents=True, exist_ok=True)

df_doccall.to_parquet(DATA_DIR / "doccall.parquet", index=False)
print(f"  ✅ doccall.parquet ({len(df_doccall):,} rows)")

df_gps.to_parquet(DATA_DIR / "gps.parquet", index=False)
print(f"  ✅ gps.parquet ({len(df_gps):,} rows)")

df_sales.to_parquet(DATA_DIR / "sales.parquet", index=False)
print(f"  ✅ sales.parquet ({len(df_sales):,} rows)")

df_address.to_parquet(DATA_DIR / "address.parquet", index=False)
print(f"  ✅ address.parquet ({len(df_address):,} rows)")

df_customer.to_parquet(DATA_DIR / "customer.parquet", index=False)
print(f"  ✅ customer.parquet ({len(df_customer):,} rows)")

df_employee.to_parquet(DATA_DIR / "employee.parquet", index=False)
print(f"  ✅ employee.parquet ({len(df_employee):,} rows)")

print("\n✅ All data extracted and cached!")

In [ ]:
# ── VERIFIKASI NAMESPACE ────────────────────────────────────────────────
# Pastikan semua ID punya prefix P1::, P2::, atau P3::

print("\n=== VERIFIKASI NAMESPACE ===")

for df_name, df in [
    ("doccall", df_doccall),
    ("sales", df_sales),
    ("customer", df_customer)
]:
    id_col = 'szId' if 'szId' in df.columns else 'szCustomerId'
    sample_ids = df[id_col].head(5).tolist()
    has_namespace = all('::' in str(id) for id in sample_ids)
    status = "✅" if has_namespace else "❌"
    print(f"{status} {df_name}: {sample_ids}")

In [ ]:
# ── RINGKASAN ───────────────────────────────────────────────────────────
print("\n" + "="*60)
print("✅ EKSTRAKSI DATA SELESAI")
print("="*60)
print(f"\n📊 Summary:")
print(f"   Kunjungan (doccall): {len(df_doccall):,} records")
print(f"   GPS tracking: {len(df_gps):,} records")
print(f"   Transaksi sales: {len(df_sales):,} records")
print(f"   Data outlet: {len(df_address):,} records")
print(f"   Master customer: {len(df_customer):,} records")
print(f"   Master employee: {len(df_employee):,} records")
print("\n🚀 Lanjut ke notebook 02_analysis.ipynb")